# Sales Feature Aggregation Tables

- `order_features.csv`: one row per order.
- `customer_features.csv`: one row per customer.
- `product_features.csv`: one row per product.
- `country_features.csv`: one row per country or market.

In simple terms, `cleaned_sales.csv` is still the transaction-detail table. The tables created here are summary tables that make the next steps easier, such as EDA, dashboard reporting, customer analysis, product analysis, and market comparison.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load the Cleaned Transaction Table

In [2]:
# 这里同样不写死绝对路径。
# 按任务要求，正式清洗文件名是 cleaned_sales.csv；cleaned_data.csv 只是兼容旧文件名的 fallback。
NOTEBOOK_DIR = Path.cwd()

candidate_cleaned_paths = [
    NOTEBOOK_DIR / "dataset" / "cleaned_sales.csv",
    NOTEBOOK_DIR / "dataset" / "cleaned_data.csv",
    NOTEBOOK_DIR.parent / "dataset" / "cleaned_sales.csv",
    NOTEBOOK_DIR.parent / "dataset" / "cleaned_data.csv",
    Path("dataset") / "cleaned_sales.csv",
    Path("dataset") / "cleaned_data.csv",
]

INPUT_PATH = next((path for path in candidate_cleaned_paths if path.exists()), None)
if INPUT_PATH is None:
    searched_paths = "\n".join(str(path) for path in candidate_cleaned_paths)
    raise FileNotFoundError(f"Cleaned dataset was not found. Searched paths:\n{searched_paths}")

# 四张聚合子表也放在同一个 dataset 文件夹下面。
DATASET_DIR = INPUT_PATH.parent
OUTPUT_DIR = DATASET_DIR / "feature_tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 这里读取清洗后的明细表。
# parse_dates 把 OrderDate 直接读成日期类型，后面才能做 first_order / last_order 这类特征。
df = pd.read_csv(INPUT_PATH, parse_dates=["OrderDate"], dtype={"CustomerID": "string"})

# 这些字段本质上是编号，不是连续数值。
# 转成字符串更符合业务含义，也能避免后面 groupby 时被误解成可以计算大小的数字。
id_columns = ["InvoiceNo", "StockCode", "CustomerID"]
for col in id_columns:
    df[col] = df[col].astype("string")

print(f"Loaded cleaned sales data from: {INPUT_PATH}")
print(f"Feature tables will be saved to: {OUTPUT_DIR}")
print(f"Cleaned sales shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

df.head()


Loaded cleaned sales data from: /Users/jia/Desktop/工作/Terra Softech/Code/dataset/cleaned_sales.csv
Feature tables will be saved to: /Users/jia/Desktop/工作/Terra Softech/Code/dataset/feature_tables
Cleaned sales shape: 392,692 rows x 10 columns


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country,OrderDate,Revenue,OrderMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55,17850,United Kingdom,2010-12-01 08:26:00,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,3.39,17850,United Kingdom,2010-12-01 08:26:00,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2.75,17850,United Kingdom,2010-12-01 08:26:00,22.00,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39,17850,United Kingdom,2010-12-01 08:26:00,20.34,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,3.39,17850,United Kingdom,2010-12-01 08:26:00,20.34,2010-12


## 2. Helper Function: Select the Most Common Value

Some fields cannot be summed directly during aggregation. For example, a customer may appear in more than one country, and a product may have repeated descriptions.  
The helper function `mode_or_first` selects the most common non-null value first. If that is not available, it uses the first valid value.

In [3]:
def mode_or_first(series: pd.Series):
    """取一个字段里最常见的值；如果没有可用值，就返回缺失。"""
    non_null = series.dropna()
    if non_null.empty:
        return pd.NA

    mode_values = non_null.mode()
    if not mode_values.empty:
        return mode_values.iloc[0]

    return non_null.iloc[0]

## 3. Order-Level Feature Table: Group by InvoiceNo

In [4]:
# 订单粒度：一行代表一个 InvoiceNo。
# 重点看这个订单花了多少钱、买了多少件、包含多少种商品。
order_features = (
    df.groupby("InvoiceNo", as_index=False)
    .agg(
        CustomerID=("CustomerID", mode_or_first),
        Country=("Country", mode_or_first),
        OrderDate=("OrderDate", "min"),
        OrderMonth=("OrderMonth", mode_or_first),
        order_line_count=("StockCode", "size"),
        order_unique_products=("StockCode", "nunique"),
        order_total_quantity=("Quantity", "sum"),
        order_total_revenue=("Revenue", "sum"),
        order_avg_unit_price=("UnitPrice", "mean"),
    )
)

# 金额类字段保留两位小数，后面展示和汇报会更清楚。
order_features["order_total_revenue"] = order_features["order_total_revenue"].round(2)
order_features["order_avg_unit_price"] = order_features["order_avg_unit_price"].round(2)

print(f"Order feature table: {order_features.shape[0]:,} orders x {order_features.shape[1]:,} columns")
order_features.head()

Order feature table: 18,532 orders x 10 columns


,InvoiceNo,CustomerID,Country,OrderDate,OrderMonth,order_line_count,order_unique_products,order_total_quantity,order_total_revenue,order_avg_unit_price
0,536365,17850,United Kingdom,2010-12-01 08:26:00,2010-12,7,7,40,139.12,3.91
1,536366,17850,United Kingdom,2010-12-01 08:28:00,2010-12,2,2,12,22.20,1.85
2,536367,13047,United Kingdom,2010-12-01 08:34:00,2010-12,12,12,83,278.73,4.85
3,536368,13047,United Kingdom,2010-12-01 08:34:00,2010-12,4,4,15,70.05,4.78
4,536369,13047,United Kingdom,2010-12-01 08:35:00,2010-12,1,1,3,17.85,5.95


## 4. Customer-Level Feature Table: Group by CustomerID


In [5]:
# 客户特征需要结合两个角度：
# 1. 从订单表看订单数量、客单价、首次/末次购买时间。
# 2. 从明细表看总购买件数、买过多少种商品、主要国家。
customer_from_orders = (
    order_features.groupby("CustomerID", as_index=False)
    .agg(
        customer_order_count=("InvoiceNo", "nunique"),
        customer_total_revenue=("order_total_revenue", "sum"),
        customer_avg_order_value=("order_total_revenue", "mean"),
        customer_first_order_date=("OrderDate", "min"),
        customer_last_order_date=("OrderDate", "max"),
        customer_active_months=("OrderMonth", "nunique"),
    )
)

customer_from_lines = (
    df.groupby("CustomerID", as_index=False)
    .agg(
        customer_total_quantity=("Quantity", "sum"),
        customer_unique_products=("StockCode", "nunique"),
        customer_line_count=("StockCode", "size"),
        customer_country_count=("Country", "nunique"),
        primary_country=("Country", mode_or_first),
    )
)

customer_features = customer_from_orders.merge(customer_from_lines, on="CustomerID", how="left")

# active_days 表示从第一次购买到最后一次购买跨了多久。
# 如果只有一次购买，这个值就是 0。
customer_features["customer_active_days"] = (
    customer_features["customer_last_order_date"] - customer_features["customer_first_order_date"]
).dt.days

customer_features["customer_total_revenue"] = customer_features["customer_total_revenue"].round(2)
customer_features["customer_avg_order_value"] = customer_features["customer_avg_order_value"].round(2)

print(f"Customer feature table: {customer_features.shape[0]:,} customers x {customer_features.shape[1]:,} columns")
customer_features.head()

Customer feature table: 4,338 customers x 13 columns


,CustomerID,customer_order_count,customer_total_revenue,customer_avg_order_value,customer_first_order_date,customer_last_order_date,customer_active_months,customer_total_quantity,customer_unique_products,customer_line_count,customer_country_count,primary_country,customer_active_days
0,12346,1,"77,183.60","77,183.60",2011-01-18 10:01:00,2011-01-18 10:01:00,1,74215,1,1,1,United Kingdom,0
1,12347,7,"4,310.00",615.71,2010-12-07 14:57:00,2011-12-07 15:52:00,7,2458,103,182,1,Iceland,365
2,12348,4,"1,797.24",449.31,2010-12-16 19:09:00,2011-09-25 13:13:00,4,2341,22,31,1,Finland,282
3,12349,1,"1,757.55","1,757.55",2011-11-21 09:51:00,2011-11-21 09:51:00,1,631,73,73,1,Italy,0
4,12350,1,334.40,334.40,2011-02-02 16:01:00,2011-02-02 16:01:00,1,197,17,17,1,Norway,0


## 5. Product-Level Feature Table: Group by StockCode


In [6]:
# 商品粒度：一行代表一个 StockCode。
# Description 不是 key，只是商品说明，所以这里取最常见的描述。
product_features = (
    df.groupby("StockCode", as_index=False)
    .agg(
        product_description=("Description", mode_or_first),
        product_total_quantity=("Quantity", "sum"),
        product_total_revenue=("Revenue", "sum"),
        product_order_count=("InvoiceNo", "nunique"),
        product_customer_count=("CustomerID", "nunique"),
        product_country_count=("Country", "nunique"),
        product_avg_unit_price=("UnitPrice", "mean"),
        product_first_sold_date=("OrderDate", "min"),
        product_last_sold_date=("OrderDate", "max"),
    )
)

product_features["product_total_revenue"] = product_features["product_total_revenue"].round(2)
product_features["product_avg_unit_price"] = product_features["product_avg_unit_price"].round(2)

# 为了预览更直观，这里按销售额从高到低看前几行。
product_features_preview = product_features.sort_values("product_total_revenue", ascending=False)

print(f"Product feature table: {product_features.shape[0]:,} products x {product_features.shape[1]:,} columns")
product_features_preview.head()

Product feature table: 3,665 products x 10 columns


,StockCode,product_description,product_total_quantity,product_total_revenue,product_order_count,product_customer_count,product_country_count,product_avg_unit_price,product_first_sold_date,product_last_sold_date
2399,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,"168,469.60",1,1,1,2.08,2011-12-09 09:15:00,2011-12-09 09:15:00
1288,22423,REGENCY CAKESTAND 3 TIER,12374,"142,264.75",1703,881,29,12.48,2010-12-01 12:27:00,2011-12-09 10:20:00
3233,85123A,WHITE HANGING HEART T-LIGHT HOLDER,36763,"100,547.45",1978,856,16,2.89,2010-12-01 08:26:00,2011-12-09 11:34:00
3219,85099B,JUMBO BAG RED RETROSPOT,46078,"85,040.54",1600,635,20,2.02,2010-12-01 09:57:00,2011-12-09 09:38:00
1997,23166,MEDIUM CERAMIC TOP STORAGE JAR,77916,"81,416.73",195,138,10,1.22,2011-01-18 10:01:00,2011-12-09 08:48:00


## 6. Country-Level Feature Table: Group by Country


In [7]:
# 国家粒度可以理解成市场维度。
# 总收入、总销量来自明细表；订单数和客单价来自订单表，这样不会把一个订单的多行商品重复当成多单。
country_from_lines = (
    df.groupby("Country", as_index=False)
    .agg(
        country_total_quantity=("Quantity", "sum"),
        country_total_revenue=("Revenue", "sum"),
        country_customer_count=("CustomerID", "nunique"),
        country_product_count=("StockCode", "nunique"),
        country_line_count=("StockCode", "size"),
    )
)

country_from_orders = (
    order_features.groupby("Country", as_index=False)
    .agg(
        country_order_count=("InvoiceNo", "nunique"),
        country_first_order_date=("OrderDate", "min"),
        country_last_order_date=("OrderDate", "max"),
    )
)

country_features = country_from_lines.merge(country_from_orders, on="Country", how="left")

# 平均订单金额 = 国家总销售额 / 国家订单数。
# 这个比直接对明细行求平均更有业务意义。
country_features["country_avg_order_value"] = (
    country_features["country_total_revenue"] / country_features["country_order_count"]
)

country_features["country_total_revenue"] = country_features["country_total_revenue"].round(2)
country_features["country_avg_order_value"] = country_features["country_avg_order_value"].round(2)

country_features_preview = country_features.sort_values("country_total_revenue", ascending=False)

print(f"Country feature table: {country_features.shape[0]:,} countries x {country_features.shape[1]:,} columns")
country_features_preview.head()

Country feature table: 37 countries x 10 columns


,Country,country_total_quantity,country_total_revenue,country_customer_count,country_product_count,country_line_count,country_order_count,country_first_order_date,country_last_order_date,country_avg_order_value
35,United Kingdom,4241305,"7,285,024.64",3920,3645,349203,16646,2010-12-01 08:26:00,2011-12-09 12:49:00,437.64
23,Netherlands,200361,"285,446.34",9,782,2359,94,2010-12-01 11:27:00,2011-12-08 12:12:00,"3,036.66"
10,EIRE,140133,"265,262.46",3,1943,7226,260,2010-12-01 14:05:00,2011-12-08 15:54:00,"1,020.24"
14,Germany,119154,"228,678.40",94,1664,9025,457,2010-12-01 13:04:00,2011-12-09 12:16:00,500.39
13,France,111428,"208,934.31",87,1522,8326,389,2010-12-01 08:45:00,2011-12-09 12:50:00,537.11


## 7. Export the Four Feature Tables

In [8]:
output_paths = {
    "order_features": OUTPUT_DIR / "order_features.csv",
    "customer_features": OUTPUT_DIR / "customer_features.csv",
    "product_features": OUTPUT_DIR / "product_features.csv",
    "country_features": OUTPUT_DIR / "country_features.csv",
}

order_features.to_csv(output_paths["order_features"], index=False)
customer_features.to_csv(output_paths["customer_features"], index=False)
product_features.to_csv(output_paths["product_features"], index=False)
country_features.to_csv(output_paths["country_features"], index=False)

# 导出后做一个小汇总，方便确认每张表的行数和保存位置。
export_summary = pd.DataFrame([
    {"table": "order_features", "rows": len(order_features), "columns": order_features.shape[1], "path": str(output_paths["order_features"])},
    {"table": "customer_features", "rows": len(customer_features), "columns": customer_features.shape[1], "path": str(output_paths["customer_features"])},
    {"table": "product_features", "rows": len(product_features), "columns": product_features.shape[1], "path": str(output_paths["product_features"])},
    {"table": "country_features", "rows": len(country_features), "columns": country_features.shape[1], "path": str(output_paths["country_features"])},
])

export_summary

,table,rows,columns,path
0,order_features,18532,10,/Users/jia/Desktop/工作/Terra Softech/Code/datas...
1,customer_features,4338,13,/Users/jia/Desktop/工作/Terra Softech/Code/datas...
2,product_features,3665,10,/Users/jia/Desktop/工作/Terra Softech/Code/datas...
3,country_features,37,10,/Users/jia/Desktop/工作/Terra Softech/Code/datas...


## 8. Validation Checks

In [9]:
# 行数检查：聚合后的表应该刚好对应原表里的唯一 key 数量。
assert len(order_features) == df["InvoiceNo"].nunique(), "Order table row count does not match unique InvoiceNo count."
assert len(customer_features) == df["CustomerID"].nunique(), "Customer table row count does not match unique CustomerID count."
assert len(product_features) == df["StockCode"].nunique(), "Product table row count does not match unique StockCode count."
assert len(country_features) == df["Country"].nunique(), "Country table row count does not match unique Country count."

# key 唯一性检查：每张子表的主键都不能重复。
assert order_features["InvoiceNo"].is_unique, "InvoiceNo should be unique in order_features."
assert customer_features["CustomerID"].is_unique, "CustomerID should be unique in customer_features."
assert product_features["StockCode"].is_unique, "StockCode should be unique in product_features."
assert country_features["Country"].is_unique, "Country should be unique in country_features."

# 文件可读取检查：确认导出的 CSV 后续可以被新的 notebook 直接使用。
for name, path in output_paths.items():
    reloaded = pd.read_csv(path, nrows=5)
    assert not reloaded.empty, f"{name} was exported but cannot be read correctly."

print("All aggregation tables were exported and validated successfully.")

All aggregation tables were exported and validated successfully.
